In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from lesson_1.ingest import load_faq_data
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import numpy as np
from elasticsearch import Elasticsearch

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

### VECTOR SEARCH with elasticsearch

In [ ]:
es = Elasticsearch("http://localhost:9200")

In [ ]:

query = "Can I still join the course after the start date?"
v_query = model.encode(query).tolist() #this has to convert to a list before feeding the query into ES
len(v_query)

In [ ]:
filter_dict = {"course": "llm-zoomcamp"}

response = es.search(
    index="embedded_faq_zoomcamp"
    ,body={
        "knn": {
            "field": "vector"
            ,"query_vector": v_query
            ,"k": 5
            ,"num_candidates": 50
            ,"filter": {"term": filter_dict}
        }
    }
)

for hit in response["hits"]["hits"]:
    print(hit["_score"])
    print(hit["_source"]["Q"], hit["_source"]["A"])
    print("---")


### VECTOR SEARCH AGAINST DATASET MANUALLY

In [ ]:
documents = load_faq_data()

texts = [doc["question"].removeprefix('Course: ') + " " + doc["answer"] for doc in documents]

#chunk the dataset (texts) by batch. There are 27 batches given texts size is 13++ and each batch is 50. Encode by batch. 
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)
len(vectors), len(vectors[0])

X = np.array(vectors)

In [ ]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

scores = X.dot(v_query)
#This is matrix-vector multiplication. 
#Each element i of scores is the cosine similarity between document i (row i of X) and v_query.

scores.shape

In [ ]:
idx = np.argmax(scores) #returns the index of the highest value in the scores array
idx, scores[idx]


In [ ]:
documents[idx]
top5 = np.argsort(-scores)[:5]
for i in top5.tolist():
    print(scores[i], documents[i])

### BASIC VECTOR SEARCH: 1 TO 1

In [ ]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)
v1

In [ ]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)
dv

In [ ]:
# we compare the query against the document using dot product:
v1.dot(dv)

In [ ]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)
v2.dot(dv)